# Benchmarks

Tianshou's algorithm implementations lead to state-of-the-art results on standard benchmarks.

An efficient parallel implementation for evaluating algorithms on MuJoCo or Atari is in `benchmark/run_benchmark.py`. It can easily be adapted for custom benchmarks as well. The reported results are thus completely reproducible.

The evaluation code uses Tianshou's integration with the [rliable](https://github.com/google-research/rliable) framework, which supports best practices for trustworthy RL evaluation. Each experiment is conducted under 5 random seeds. We report the interquartile mean (IQM) and 95% confidence intervals over these seeds.

[Download the Ant-v4 results (JSON)](../_static/js/mujoco/benchmark/Ant-v4/results.json)

In [ ]:
import json
from html import escape
from pathlib import Path

import altair as alt
from IPython.display import IFrame


def benchmark_chart(suite: str, environment: str) -> alt.LayerChart:
    results_path = Path("../_static/js") / suite / "benchmark" / environment / "results.json"
    results = json.loads(results_path.read_text(encoding="utf-8"))
    step_limit = alt.param(
        name="Range",
        value=1_000_000,
        bind=alt.binding_range(min=10_000, max=10_000_000, step=10_000, name="Env step: "),
    )
    base = (
        alt.Chart(alt.Data(values=results))
        .transform_filter(alt.datum.env_step <= step_limit)
        .encode(
            x=alt.X("env_step:Q", title="Env step"),
            color="agent:N",
            tooltip=[
                alt.Tooltip("env_step:Q", title="Env step"),
                alt.Tooltip("agent:N"),
                alt.Tooltip("iqm:Q", title="Return"),
            ],
        )
    )
    return_title = "Return Interquartile Mean (5 seeds)"
    confidence_interval = base.mark_area(opacity=0.3).encode(
        y=alt.Y("iqm_confidence_interval[0]:Q", title=return_title),
        y2="iqm_confidence_interval[1]:Q",
    )
    iqm = base.mark_line().encode(y=alt.Y("iqm:Q", title=return_title))
    return (
        (confidence_interval + iqm)
        .add_params(step_limit)
        .properties(title=environment, width="container", height=400)
    )

In [ ]:
chart = benchmark_chart("mujoco", "Ant-v4")
# Isolate the bundled Vega scripts from the documentation theme.
IFrame(
    src="about:blank",
    width="100%",
    height=520,
    extras=['title="Ant-v4 benchmark"', f'srcdoc="{escape(chart.to_html(inline=True))}"'],
)